<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# Model Evaluation and Selection Review Questions {#sec-model-eval-selection-questions}

These questions accompany `005_evaluation_and_model_selection_v3.ipynb`. Answers are hidden in collapsible sections.


### Question 1

An evaluation dataset $\mathcal D_{\mathrm{eval}}$ did not influence a fitted model $\hat f$, but it was collected from a population different from the one where the model will be deployed. Does $M(\mathcal D_{\mathrm{eval}},\hat f)$ estimate the desired deployment performance? 

<details>
<summary>Answer</summary>

Not necessarily. Independence from the fitted model prevents reuse of evaluation information, but the evaluation observations must also represent the target population. Otherwise the metric estimates performance on the evaluation population rather than on the deployment population. Thus we need both separation from model building and a distribution appropriate to the intended use.

</details>


### Question 2

A student argues: “The fitting algorithm minimized training error, so training error is the most direct measure of how good the fitted model is.” Diagnose the mistake. 

<details>
<summary>Answer</summary>

Minimizing training error makes it a direct measure of fit to the observed sample, but that is why it is not neutral evidence about new data. The algorithm searched for a function that performs well on those observations, rewarding improvements caused by reproducible signal and improvements caused by sample-specific variation. Generalization asks which improvements persist on new observations, so it requires data that did not determine the fit.

</details>


### Question 3

Let $\hat f_d$ be the least-squares fit among polynomials of degree at most $d$, using a fixed training dataset. Prove that
$$
\operatorname{MSE}_{\mathrm{train}}(\hat f_{d+1})
\le
\operatorname{MSE}_{\mathrm{train}}(\hat f_d).
$$
Does the same inequality necessarily hold for evaluation MSE?

<details>
<summary>Answer</summary>

Every polynomial of degree at most $d$ is also a polynomial of degree at most $d+1$: set the coefficient of $x^{d+1}$ to zero. Thus the degree-$(d+1)$ optimization searches over a set containing every degree-$d$ candidate. Its minimum training MSE cannot be larger.

No analogous conclusion follows for evaluation MSE. The additional coefficient is selected using the training sample and may fit sample-specific noise that does not recur in the evaluation data.

</details>


### Question 4

In the lecture's cartoon of training and evaluation error versus model complexity, describe the typical behavior in three regions: low, intermediate, and high complexity. Identify underfitting and overfitting, and explain why the evaluation curve's U shape is a pattern rather than a theorem.

<details>
<summary>Answer</summary>

At low complexity, the class cannot represent enough of the signal, so both errors are high: this is underfitting. At intermediate complexity, the model captures more signal and evaluation error is often near its minimum. At high complexity, training error can continue downward while evaluation error rises because the model increasingly fits sample-specific variation: this is overfitting.

The U shape is not guaranteed because observed errors contain sampling noise and different data-generating processes, model classes, and fitting procedures can produce different curves.

</details>


### Question 5

An analyst randomly splits observations into $\mathcal D_{\mathrm{train}}$ and $\mathcal D_{\mathrm{eval}}$, but computes means and standard deviations using all observations before the split. The model uses those values to standardize its inputs. Is the subsequent evaluation reasonable? 
<details>
<summary>Answer</summary>

No. The fitted model depends on summary information from $\mathcal D_{\mathrm{eval}}$, so the evaluation observations have influenced model building. Fit the standardization parameters using only $\mathcal D_{\mathrm{train}}$, then apply that fixed transformation to $\mathcal D_{\mathrm{eval}}$. In cross-validation, repeat this fitting step inside every training fold; a pipeline can enforce that order.

</details>


### Question 6

Five-fold cross-validation is used. How many times is each observation used for evaluation and for training?

<details>
<summary>Answer</summary>

Each observation belongs to one evaluation fold, so it is used for evaluation once. It belongs to the training portion for each of the other four fits, so it is used for training four times.

</details>


### Question 7

A false argument says: “Fold $\mathcal F_j$ is omitted when fitting $\hat f^{(-j)}$, so the $k$ cross-validation scores $e_1,\ldots,e_k$ are independent.” Identify the false step.

<details>
<summary>Answer</summary>

Omitting $\mathcal F_j$ from its corresponding fit supports evaluating $\hat f^{(-j)}$ on that fold. It does not make scores from different folds independent. The training sets $\mathcal D\setminus\mathcal F_j$ overlap heavily, and every score is a function of the same original dataset. Consequently the fold scores are generally dependent, and their sample standard deviation is not automatically a standard error for their mean.

</details>


### Question 8

After $k$-fold CV, a collaborator asks which fitted function
$$
\hat f^{(-1)},\ldots,\hat f^{(-k)}
$$
should be deployed. What is the correct response?

<details>
<summary>Answer</summary>

None of the fold-specific functions is ordinarily deployed. They are temporary fits used to estimate the performance of the fixed fitting procedure $A$. Once the procedure is settled, fit
$$
\hat f_{\mathrm{final}}=A(\mathcal D)
$$
on all available training data and deploy that result. CV supplied evidence about the procedure, not an independent evaluation of this particular final fit.

</details>


### Question 9

A dataset contains $N=200$ observations and is evaluated using five-fold CV. At what training-sample size is the procedure directly evaluated in each fold? 

<details>
<summary>Answer</summary>

Each fit uses four of the five folds, hence
$$
200\left(\frac45\right)=160
$$
training observations. CV evaluates several outputs of the procedure trained on 160 observations. The final model uses all 200 observations and is a different fitted function, so the CV score is not its direct test error. It is an estimate of procedure-level performance at the fold training size and is often informative about the final full-data fit.

</details>


### Question 10

Three polynomial procedures have validation MSEs

| Degree $d$ | 1 | 3 | 5 |
|---:|---:|---:|---:|
| $e_d$ | 4.8 | 2.1 | 2.7 |

Compute the selected degree $\hat d$. 

<details>
<summary>Answer</summary>

The minimum validation MSE is $2.1$, so
$$
\hat d=\arg\min_{d\in\{1,3,5\}}e_d=3.
$$

</details>


### Question 11

For every fixed $\lambda$, the candidate
$$
\hat f_\lambda=A_\lambda(\mathcal D_{\mathrm{train}})
$$
is evaluated on separate validation data. Explain why $e_\lambda=M(\mathcal D_{\mathrm{val}},\hat f_\lambda)$ can reasonably estimate that candidate's performance while the winning score $e_{\hat\lambda}$ is usually optimistic after selection.

<details>
<summary>Answer</summary>

For a fixed $\lambda$, validation observations did not determine $\hat f_\lambda$, so their metric provides fresh-data evidence for that candidate. Selection changes the situation: $\hat\lambda$ is chosen because $e_{\hat\lambda}$ is the most favorable among several noisy estimates. The winning value can be low because of genuinely better performance and because its validation noise happened to be favorable. It has therefore lost its role as an independent evaluation of the selected result.

</details>


### Question 12

Two candidates have the same true error, $1$. Their validation errors are
$$
e_1=1+\varepsilon_1,
\qquad
e_2=1+\varepsilon_2,
$$
where $\varepsilon_1$ and $\varepsilon_2$ are independent and each equals $-0.2$ or $0.2$ with probability $1/2$. Compute $\mathbb E[\min(e_1,e_2)]$ and interpret the result.

<details>
<summary>Answer</summary>

The four pairs of noise values are equally likely. The minimum validation error is $0.8$ in three cases and $1.2$ only when both noise values equal $0.2$. Hence
$$
\mathbb E[\min(e_1,e_2)]
=\frac34(0.8)+\frac14(1.2)=0.9.
$$
Each candidate's validation error is centered at its true error, but selecting the smaller estimate produces an expected reported value below the common true error. This is selection optimism.

</details>


### Question 13

Why can searching a larger candidate set $\Lambda$, or repeatedly changing $\Lambda$ after inspecting validation results, increase validation-set overfitting? Give an answer in terms of noisy performance estimates and the larger procedure's capacity.

<details>
<summary>Answer</summary>

Each candidate has a noisy validation estimate. A larger search creates more opportunities to encounter an unusually favorable estimate and select it. Revising the search after inspecting results adapts the candidate set itself to the validation data. Both actions increase the capacity of the meta-procedure to exploit validation noise, so the winning score can become more optimistic even if the selected model's true performance does not improve.

</details>


### Question 14

In a three-way split, state the role of each dataset:
$$
\mathcal D_{\mathrm{train}},\qquad
\mathcal D_{\mathrm{val}},\qquad
\mathcal D_{\mathrm{test}}.
$$
For each one, say whether it may influence candidate parameters, the selected $\hat\lambda$, and the final reported performance estimate.

<details>
<summary>Answer</summary>

$\mathcal D_{\mathrm{train}}$ fits candidate parameters. $\mathcal D_{\mathrm{val}}$ compares candidates and therefore influences $\hat\lambda$. Both are model-building data. After selection, they may be combined to refit the selected procedure.

$\mathcal D_{\mathrm{test}}$ should influence neither candidate parameters nor $\hat\lambda$. It is used only after model building is complete to produce the final reported performance estimate.

</details>


### Question 15

After selecting $\hat\lambda$ using validation data, why is it reasonable to refit $A_{\hat\lambda}$ on
$$
\mathcal D_{\mathrm{train}}\cup\mathcal D_{\mathrm{val}}
$$
before evaluating on $\mathcal D_{\mathrm{test}}$? Why does this not invalidate the test evaluation?

<details>
<summary>Answer</summary>

Validation data have already participated in model building by selecting $\hat\lambda$. Once that choice is fixed, combining them with the original training data gives the final coefficient fit more observations. This does not invalidate test evaluation because $\mathcal D_{\mathrm{test}}$ remains absent from fitting, validation, selection, and refitting.

</details>


### Question 16

Before splitting its data, an analyst selects the 20 features having the largest correlations with the response. The analyst then performs a train/validation/test split and reports test error. Identify the leakage and give the correct workflow.

<details>
<summary>Answer</summary>

Feature selection used every response, including test responses, so the selected feature set—and hence the final model—depends on test data. The later split cannot remove that dependence.

First reserve the test set. Select features using only model-building data. If validation or CV is used, repeat response-based feature selection inside each training split or fold. Apply the resulting selected-feature rule to the corresponding validation data, and leave test data untouched until the final evaluation.

</details>
